#Recommender Net with Label Encoder Movielens


In [ ]:
!pip install gdown pandas scikit-learn torch tqdm --quiet

import gdown
file_urls = {
    'movies':  "https://drive.google.com/uc?id=1z1UUE-TRQhaiydYF-NaoLLJO9Qb0JgXW",
    'ratings': "https://drive.google.com/uc?id=1v-DFz7NKN_nqJC_KSKILHsOmG2-dLHtv",
    'links':   "https://drive.google.com/uc?id=1nCG7xhkTcU15jKZnqA6twR4IHn_cn6qe",
    'tags':    "https://drive.google.com/uc?id=12RifUdEgvsdIKY-ofJhYHi4XdUoDYOnk",
}
for name, url in file_urls.items():
    gdown.download(url, f"{name}.csv", quiet=False)

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
links = pd.read_csv("links.csv")
tags = pd.read_csv("tags.csv")

tags['tag'] = tags['tag'].fillna('').astype(str)
tags_agg = tags.groupby("movieId")['tag'].apply(lambda x: '|'.join(set(x))).reset_index()
movie_features = movies.merge(links, on='movieId', how='left').merge(tags_agg, on='movieId', how='left')
data = ratings.merge(movie_features, on='movieId', how='left')

user_enc = LabelEncoder()
item_enc = LabelEncoder()
data['user_enc'] = user_enc.fit_transform(data['userId'])
data['item_enc'] = item_enc.fit_transform(data['movieId'])

data['genres'] = data['genres'].fillna('').apply(lambda x: x.split('|'))
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(data['genres'])

X = {
    'user': data['user_enc'].values,
    'item': data['item_enc'].values,
    'genre': genre_encoded
}
y = data['rating'].values.astype(np.float32)

train_idx, val_idx = train_test_split(np.arange(len(y)), test_size=0.1, random_state=42)
X_train = {k: v[train_idx] for k, v in X.items()}
X_val = {k: v[val_idx] for k, v in X.items()}
y_train, y_val = y[train_idx], y[val_idx]

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

class MovieLensDataset(Dataset):
    def __init__(self, X, y):
        self.user = torch.tensor(X['user'], dtype=torch.long)
        self.item = torch.tensor(X['item'], dtype=torch.long)
        self.genre = torch.tensor(X['genre'], dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.user)

    def __getitem__(self, idx):
        return self.user[idx], self.item[idx], self.genre[idx], self.y[idx]

class RecommenderNet(nn.Module):
    def __init__(self, n_users, n_items, genre_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, 64)
        self.item_emb = nn.Embedding(n_items, 64)
        self.fc1 = nn.Linear(64 + 64 + genre_dim, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, user, item, genre):
        u = self.user_emb(user)
        i = self.item_emb(item)
        x = torch.cat([u, i, genre], dim=1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x).squeeze()

n_users = data['user_enc'].nunique()
n_items = data['item_enc'].nunique()
genre_dim = genre_encoded.shape[1]

model = RecommenderNet(n_users, n_items, genre_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_loader = DataLoader(MovieLensDataset(X_train, y_train), batch_size=512, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(5):
    for user, item, genre, target in tqdm(train_loader):
        user, item, genre, target = user.to(device), item.to(device), genre.to(device), target.to(device)
        optimizer.zero_grad()
        preds = model(user, item, genre)
        loss = loss_fn(preds, target)
        loss.backward()
        optimizer.step()

# Predict regression output
model.eval()
with torch.no_grad():
    user = torch.tensor(X_val['user'], dtype=torch.long).to(device)
    item = torch.tensor(X_val['item'], dtype=torch.long).to(device)
    genre = torch.tensor(X_val['genre'], dtype=torch.float32).to(device)
    preds_val = model(user, item, genre).cpu().numpy()

from collections import defaultdict
from sklearn.metrics import mean_squared_error, mean_absolute_error

val_data = data.iloc[val_idx]

relevant_items_by_user = (
    val_data[val_data["rating"] >= 4]
    .groupby("user_enc")["item_enc"]
    .apply(set)
    .to_dict()
)

all_items = np.unique(data['item_enc'])
top_k_preds = []
ground_truth = []
K = 10

model.eval()
with torch.no_grad():
    for user in np.unique(X_val['user']):
        user_tensor = torch.tensor([user] * len(all_items), dtype=torch.long).to(device)
        item_tensor = torch.tensor(all_items, dtype=torch.long).to(device)
        genre_tensor = torch.tensor(
            np.repeat([X_val['genre'][X_val['user'] == user][0]], len(all_items), axis=0),
            dtype=torch.float32
        ).to(device)

        scores = model(user_tensor, item_tensor, genre_tensor).cpu().numpy()
        top_items = all_items[np.argsort(scores)[-K:][::-1]]

        top_k_preds.append(list(top_items))
        ground_truth.append(list(relevant_items_by_user.get(user, [])))

#Metric functions
def precision_at_k(y_true, y_pred, k):
    return np.mean([len(set(pred[:k]) & set(true)) / k for pred, true in zip(y_pred, y_true)])

def recall_at_k(y_true, y_pred, k):
    return np.mean([len(set(pred[:k]) & set(true)) / len(true) if true else 0 for pred, true in zip(y_pred, y_true)])

def ndcg_at_k(y_true, y_pred, k):
    def dcg(rel):
        return sum(r / np.log2(i + 2) for i, r in enumerate(rel))
    scores = []
    for true, pred in zip(y_true, y_pred):
        rel = [1 if p in true else 0 for p in pred[:k]]
        ideal = sorted(rel, reverse=True)
        score = dcg(rel) / dcg(ideal) if dcg(ideal) > 0 else 0
        scores.append(score)
    return np.mean(scores)

def mean_average_precision_at_k(y_true, y_pred, k):
    average_precisions = []
    for true, pred in zip(y_true, y_pred):
        hits, sum_precisions = 0, 0
        for i, p in enumerate(pred[:k]):
            if p in true:
                hits += 1
                sum_precisions += hits / (i + 1)
        average_precisions.append(sum_precisions / min(len(true), k) if true else 0)
    return np.mean(average_precisions)

#Compute and print results
rmse = np.sqrt(mean_squared_error(y_val, preds_val))
mae = mean_absolute_error(y_val, preds_val)
precision = precision_at_k(ground_truth, top_k_preds, K)
recall = recall_at_k(ground_truth, top_k_preds, K)
ndcg = ndcg_at_k(ground_truth, top_k_preds, K)
mapk = mean_average_precision_at_k(ground_truth, top_k_preds, K)

import pandas as pd
metrics_table_corrected = pd.DataFrame({
    "Model": ["Recommender Net"],
    "RMSE": [rmse],
    "MAE": [mae],
    "Precision@10": [precision],
    "Recall@10": [recall],
    "NDCG@10": [ndcg],
    "MAP@10": [mapk]
})

metrics_table_corrected


RMSE	0.814063

MAE	0.616479

Precision@10	0.006991

Recall@10	0.011049

NDCG@10	0.034319

MAP@10 0.004726



#Content Based BM25 Movielens100K

In [ ]:
import pandas as pd
import numpy as np
!pip install rank_bm25
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')  # corrected from 'punkt_tab'

# Load datasets
movies = pd.read_csv('movies.csv')
tags = pd.read_csv('tags.csv')
ratings = pd.read_csv('ratings.csv')

# Merge tags into movies
tagged = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
movies = pd.merge(movies, tagged, on='movieId', how='left')
movies['tag'] = movies['tag'].fillna('')

# Combine genres and tags into one metadata string
movies['metadata'] = movies['genres'].str.replace('|', ' ', regex=False) + ' ' + movies['tag']

# Tokenize metadata for BM25
tokenized_corpus = movies['metadata'].apply(lambda x: word_tokenize(x.lower())).tolist()
bm25 = BM25Okapi(tokenized_corpus)
movie_indices = pd.Series(movies.index, index=movies['movieId'])

# Merge ratings into movies (average rating per movie)
avg_ratings = ratings.groupby('movieId')['rating'].mean().reset_index()
avg_ratings.columns = ['movieId', 'avg_rating']
movies = pd.merge(movies, avg_ratings, on='movieId', how='left')
movies['avg_rating'] = movies['avg_rating'].fillna(0)

# Recommendation function based on BM25 and rating boost
def content_based_recommendations(movie_id, top_n=10, rating_weight=0.2):
    idx = movie_indices[movie_id]
    query = tokenized_corpus[idx]
    bm25_scores = bm25.get_scores(query)

    # Enhance BM25 score with average rating
    enhanced_scores = []
    for i, score in enumerate(bm25_scores):
        rating_boost = rating_weight * movies.iloc[i]['avg_rating'] / 5
        enhanced_scores.append((i, score + rating_boost))

    enhanced_scores = sorted(enhanced_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices_rec = [i[0] for i in enhanced_scores]
    return movie_indices_rec

# Evaluation with NDCG
def evaluate_ndcg(sample_size=100, top_n=10):
    sample = ratings[ratings['rating'] >= 4.0].groupby('userId').sample(n=1, random_state=42)
    ndcg_scores = []

    for _, row in sample.iterrows():
        movie_id = row['movieId']
        relevant_idx = movie_indices.get(movie_id)
        if pd.isna(relevant_idx):
            continue

        recommendations = content_based_recommendations(movie_id, top_n=top_n)

        y_true = np.zeros((1, len(movies)))
        y_score = np.zeros((1, len(movies)))

        y_true[0, int(relevant_idx)] = 1
        for rank, idx in enumerate(recommendations):
            y_score[0, idx] = top_n - rank

        score = ndcg_score(y_true, y_score)
        ndcg_scores.append(score)

    return np.mean(ndcg_scores) if ndcg_scores else 0

# Example usage
ndcg_result = evaluate_ndcg(sample_size=100, top_n=10)
print(f"Average NDCG@10 Score using BM25: {ndcg_result:.4f}")

from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt

def additional_metrics(sample_size=100, top_k=10):
    sample = ratings[ratings['rating'] >= 4.0].groupby('userId').sample(n=1, random_state=42)

    y_true_all = []
    y_pred_all = []

    ndcg_scores = []
    precision_scores = []
    recall_scores = []

    for _, row in sample.iterrows():
        user_movie_id = row['movieId']
        user_rating = row['rating']
        relevant_idx = movie_indices.get(user_movie_id)

        if pd.isna(relevant_idx):
            continue

        rec_indices = content_based_recommendations(user_movie_id, top_n=top_k)

        for idx in rec_indices:
            y_pred_all.append(movies.iloc[idx]['avg_rating'])
            y_true_all.append(user_rating)  # Ground truth is the high rating (≥4.0)

        # Binary relevance vector
        y_true_bin = np.zeros(len(movies))
        y_true_bin[int(relevant_idx)] = 1

        y_score_bin = np.zeros(len(movies))
        for rank, idx in enumerate(rec_indices):
            y_score_bin[idx] = top_k - rank

        ndcg_scores.append(ndcg_score([y_true_bin], [y_score_bin]))

        # Precision & Recall @K
        hits = 1 if int(relevant_idx) in rec_indices else 0
        precision_scores.append(hits / top_k)
        recall_scores.append(hits / 1)  # Only one relevant item in this simplified case

    # Error Metrics
    rmse = sqrt(mean_squared_error(y_true_all, y_pred_all)) if y_true_all else 0
    mae = mean_absolute_error(y_true_all, y_pred_all) if y_true_all else 0
    mape = np.mean(np.abs((np.array(y_true_all) - np.array(y_pred_all)) / np.array(y_true_all))) * 100 if y_true_all else 0

    return {
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        f'NDCG@{top_k}': np.mean(ndcg_scores),
        f'Precision@{top_k}': np.mean(precision_scores),
        f'Recall@{top_k}': np.mean(recall_scores)
    }

metrics_result = additional_metrics(sample_size=100, top_k=10)
print("\nEvaluation Metrics:")
for metric, value in metrics_result.items():
    print(f"{metric}: {value:.4f}")


Additional Evaluation Metrics:

RMSE: 1.0541

MAE: 0.8178

MAPE: 18.1852

NDCG@10: 0.1771

Precision@10: 0.0171

Recall@10: 0.1708

#Content based Recommender Net TF-IDF Movielens

In [ ]:
# Content-Based Recommender System using TF-IDF and Ratings (MovieLens)

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score

# Load datasets
movies = pd.read_csv('movies.csv')
tags = pd.read_csv('tags.csv')
ratings = pd.read_csv('ratings.csv')

# Merge tags into movies
tagged = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
movies = pd.merge(movies, tagged, on='movieId', how='left')
movies['tag'] = movies['tag'].fillna('')

# Combine genres and tags into one metadata string
movies['metadata'] = movies['genres'].str.replace('|', ' ', regex=False) + ' ' + movies['tag']

# TF-IDF vectorization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['metadata'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
movie_indices = pd.Series(movies.index, index=movies['movieId'])

# Merge ratings into movies (average rating per movie)
avg_ratings = ratings.groupby('movieId')['rating'].mean().reset_index()
avg_ratings.columns = ['movieId', 'avg_rating']
movies = pd.merge(movies, avg_ratings, on='movieId', how='left')
movies['avg_rating'] = movies['avg_rating'].fillna(0)

# Recommendation function based on TF-IDF similarity and rating boost
def content_based_recommendations(movie_id, top_n=10, rating_weight=0.2):
    idx = movie_indices[movie_id]
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Enhance similarity with average ratings
    enhanced_scores = []
    for i, score in sim_scores:
        rating_boost = rating_weight * movies.iloc[i]['avg_rating'] / 5  # normalize rating to 0-1 scale
        enhanced_scores.append((i, score + rating_boost))

    enhanced_scores = sorted(enhanced_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices_rec = [i[0] for i in enhanced_scores]
    return movie_indices_rec
from sklearn.metrics import ndcg_score
from collections import defaultdict

# Evaluation with NDCG, Precision, Recall, MAP
def evaluate_all_metrics(sample_size=100, top_n=10):
    sample = ratings[ratings['rating'] >= 4.0].groupby('userId').sample(n=1, random_state=42)

    ndcg_scores = []
    precisions = []
    recalls = []
    average_precisions = []

    for _, row in sample.iterrows():
        movie_id = row['movieId']
        relevant_idx = movie_indices.get(movie_id)
        if pd.isna(relevant_idx):
            continue

        recommendations = content_based_recommendations(movie_id, top_n=top_n)

        # NDCG calculation
        y_true = np.zeros((1, len(movies)))
        y_score = np.zeros((1, len(movies)))
        y_true[0, int(relevant_idx)] = 1
        for rank, idx in enumerate(recommendations):
            y_score[0, idx] = top_n - rank

        ndcg = ndcg_score(y_true, y_score)
        ndcg_scores.append(ndcg)

        # Precision, Recall, MAP calculation
        relevant_set = {int(relevant_idx)}
        recommended_set = set(recommendations)

        hits = relevant_set & recommended_set
        precision = len(hits) / top_n
        recall = len(hits) / len(relevant_set)  # always 1 relevant item here
        precisions.append(precision)
        recalls.append(recall)

        # MAP
        ap = 0
        for i, rec in enumerate(recommendations):
            if rec in relevant_set:
                ap = 1 / (i + 1)
                break
        average_precisions.append(ap)

    return {
        "Precision@10": np.mean(precisions),
        "Recall@10": np.mean(recalls),
        "NDCG@10": np.mean(ndcg_scores),
        "MAP@10": np.mean(average_precisions)
    }

# Example usage
results = evaluate_all_metrics(sample_size=100, top_n=10)
metrics_table = pd.DataFrame([{
    "Model": "TF-IDF Content-Based",
    "Precision@10": results["Precision@10"],
    "Recall@10": results["Recall@10"],
    "NDCG@10": results["NDCG@10"],
    "MAP@10": results["MAP@10"]
}])

print(metrics_table)

Precision@10  0.015435   

Recall@10     0.154351

NDCG@10       0.164802

MAP@10.       0.072563


#SASRec and SVD Hybrid Recommender

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import ndcg_score, precision_score, recall_score
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

ratings = pd.read_csv('ratings.csv')
movies_df = pd.read_csv('movies.csv')
ratings['rating'] /= 5

# Build user-item matrix
user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Prepare SVD model
svd = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_.T
reconstructed = np.dot(user_factors, item_factors.T)

# Recommend top-N items for user 1
user_index = 0  # userId = 1
user_ratings = user_item_matrix.iloc[user_index].values
reconstructed_scores = reconstructed[user_index]

# Filter out already seen movies
seen = user_ratings > 0
reconstructed_scores[seen] = -np.inf

# Get top-N recommendations
top_n = 10
top_indices = np.argsort(reconstructed_scores)[-top_n:][::-1]
top_movie_ids = user_item_matrix.columns[top_indices]
movie_titles = dict(zip(movies_df['movieId'], movies_df['title']))

print("SVD Recommendations for user 1:")
for i, mid in enumerate(top_movie_ids, 1):
    print(f"{i}. {movie_titles.get(mid, f'Unknown ({mid})')}")

# Evaluate SVD model

def evaluate_svd(user_item_matrix, reconstructed, top_k=10, sample_users=100):
    hit, ndcg, precision, recall = [], [], [], []
    users_with_enough = user_item_matrix[user_item_matrix.gt(0).sum(axis=1) > 5]
    sampled_users = users_with_enough.sample(n=min(sample_users, len(users_with_enough)), random_state=42)

    for user_id, row in sampled_users.iterrows():
        true_items = row[row > 0].index.tolist()
        if len(true_items) < 2:
            continue
        held_out = true_items[-1]
        user_seen = row.copy()
        user_seen[held_out] = 0

        scores = reconstructed[user_id - 1].copy()
        scores[user_seen > 0] = -np.inf

        top_preds = np.argsort(scores)[::-1][:top_k]
        held_out_idx = user_item_matrix.columns.get_loc(held_out)

        hit.append(int(held_out_idx in top_preds))
        rel = [1 if i == held_out_idx else 0 for i in top_preds]
        ndcg.append(ndcg_score([rel], [list(range(top_k, 0, -1))]))

        precision.append(np.sum(rel) / top_k)
        recall.append(np.sum(rel))  # 1 relevant item

    results = pd.DataFrame({
        'Metric': ['Hit@10', 'NDCG@10', 'Precision@10', 'Recall@10'],
        'Score': [np.mean(hit), np.mean(ndcg), np.mean(precision), np.mean(recall)]
    })
    print("\nEvaluation Metrics Table:")
    print(results.to_string(index=False))

# Run evaluation
evaluate_svd(user_item_matrix, reconstructed, top_k=10, sample_users=100)


SVD Recommendations for user 1:
1. Die Hard (1988)
2. Godfather: Part II, The (1974)
3. Godfather, The (1972)
4. Stand by Me (1986)
5. Jaws (1975)
6. Terminator 2: Judgment Day (1991)
7. Breakfast Club, The (1985)
8. Aliens (1986)
9. Ferris Bueller's Day Off (1986)
10. Christmas Story, A (1983)

Evaluation Metrics Table:

    Metric    Score

    Hit@10 0.150000

    NDCG@10 0.120666

    Precision@10 0.015000

    Recall@10 0.150000

#NCF Recommenders Library Movielens

In [ ]:
# Neural Collaborative Filtering (NCF) with MovieLens 1M

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from recommenders.datasets import movielens
from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.utils.constants import SEED
from recommenders.evaluation.python_evaluation import map_at_k, ndcg_at_k, precision_at_k, recall_at_k
from recommenders.utils.timer import Timer
from recommenders.datasets.python_splitters import python_chrono_split
from recommenders.evaluation.python_evaluation import get_top_k_items

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import os
import tempfile

print("Downloading MovieLens 1M dataset...")
df = movielens.load_pandas_df(size="1m")

print("Data preview:")
print(df.head())

ratings = df.rename(columns={"userID": "userID", "itemID": "itemID", "rating": "rating"})

print("Splitting data into train and test sets...")
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=SEED)
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

train_df = train_df.sort_values(by="userID")
test_df = test_df.sort_values(by="userID")

with tempfile.TemporaryDirectory() as temp_dir:
    train_path = os.path.join(temp_dir, "train.csv")
    test_path = os.path.join(temp_dir, "test.csv")
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Converting to NCFDataset...")
    data = NCFDataset(train_file=train_path, test_file=test_path, seed=SEED)

    print("Initializing NCF model...")
    ncf_model = NCF(
        n_users=data.n_users,
        n_items=data.n_items,
        model_type="NeuMF",  # Options: "NeuMF", "GMF", "MLP"
        learning_rate=0.001,
        batch_size=256,
        verbose=10,
        seed=SEED,
    )

    print("Training the model...")
    with Timer() as train_time:
        ncf_model.fit(data)

    print(f"Training took: {train_time}")

    print("Generating top-K predictions...")
    top_k = 10

    # Only use users and items seen during training
    known_users = set(train_df["userID"].unique())
    known_items = set(train_df["itemID"].unique())

    test_users = test_df[test_df.userID.isin(known_users)].userID.unique()
    all_items = df[df.itemID.isin(known_items)].itemID.unique()

    user_input, item_input = [], []
    for u in test_users:
        user_input.extend([u] * len(all_items))
        item_input.extend(all_items)

    scores = ncf_model.predict(user_input, item_input, is_list=True)
    all_preds = pd.DataFrame({"userID": user_input, "itemID": item_input, "prediction": scores})
    all_preds.rename(columns={"prediction": "rating"}, inplace=True)
    top_k_preds = get_top_k_items(all_preds, k=top_k)
    top_k_preds["prediction"] = top_k_preds["rating"]  # copy predicted score
    top_k_preds["rating"] = 1  # dummy relevancy score
    top_k_preds = top_k_preds[["userID", "itemID", "rating", "prediction"]].astype({
        "userID": int, "itemID": int, "rating": int, "prediction": float
    })

    test_df_filtered = test_df[test_df.userID.isin(test_users) & test_df.itemID.isin(known_items)].copy()
    test_df_filtered = test_df_filtered[["userID", "itemID", "rating"]].astype({
        "userID": int, "itemID": int, "rating": int
    })

    print("Evaluating model (Top-K metrics)...")
    k = 10
    print(f"MAP@{k}: {map_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=k):.4f}")
    print(f"NDCG@{k}: {ndcg_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=k):.4f}")
    print(f"Precision@{k}: {precision_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=k):.4f}")
    print(f"Recall@{k}: {recall_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=k):.4f}")





MAP@10: 0.0558

NDCG@10: 0.1354

Precision@10: 0.1248

Recall@10: 0.0728

#LightGBM Embeddings Google Maps


In [ ]:
!pip install -q lightgbm tqdm sentence-transformers

from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import json
import torch
import re
import random
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
from collections import defaultdict

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

file_path = '/content/drive/MyDrive/review-South_Carolina_10.json'
with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]
df = pd.DataFrame(data).sample(frac=0.8, random_state=42).reset_index(drop=True)

# Text cleaning
def clean_text(text, max_words=80):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower()
    return ' '.join(text.split()[:max_words])

df['text'] = df['text'].fillna('').apply(clean_text)

# Feature engineering
df['text_length'] = df['text'].str.len()
df['num_exclamations'] = df['text'].str.count('!')
df['time'] = pd.to_datetime(df['time'], unit='ms', errors='coerce')
df['hour'] = df['time'].dt.hour
df['weekday'] = df['time'].dt.weekday
df['month'] = df['time'].dt.month
df['time_of_day'] = df['hour'].apply(
    lambda h: 'morning' if 5 <= h < 12 else
              'afternoon' if 12 <= h < 17 else
              'evening' if 17 <= h < 21 else 'night'
).astype('category')
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0)
df['days_since_review'] = (df['time'].max() - df['time']).dt.days
df['user_id'] = df['user_id'].astype('category')
df['gmap_id'] = df['gmap_id'].astype('category')
y = df['rating'] - 1

# Structured features
simple_features = [
    'hour', 'weekday', 'month', 'is_weekend',
    'time_of_day', 'days_since_review',
    'text_length', 'num_exclamations'
]
X_simple = df[simple_features].reset_index(drop=True)
categorical_features = ['time_of_day']

# Embedding setup
embedding_path = '/content/drive/MyDrive/full_embeddings.npy'
texts = df['text'].tolist()
total_rows = len(texts)
embedding_dim = 384

if os.path.exists(embedding_path):
    print("Embedding file already exists, skipping recomputation.")
else
    embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
    embedder.half()
    batch_size = 1024
    embedding_dim = embedder.encode(["test"], device='cuda', convert_to_numpy=True).shape[1]
    embedding_array = np.memmap(embedding_path, dtype='float16', mode='w+', shape=(total_rows, embedding_dim))

    print("Embedding text to Google Drive...")
    cursor = 0
    for i in tqdm(range(0, total_rows, batch_size), desc="Embedding"):
        batch = texts[i:i+batch_size]
        emb = embedder.encode(batch, batch_size=batch_size, device='cuda', convert_to_numpy=True)
        embedding_array[cursor:cursor+len(emb)] = emb.astype('float16')
        cursor += len(emb)
        del batch, emb
        gc.collect()
        torch.cuda.empty_cache()

    embedding_array.flush()
    del embedding_array
    gc.collect()
    print("✅ Embeddings saved.")

embedding_loaded = np.memmap(embedding_path, dtype='float16', mode='r', shape=(total_rows, embedding_dim))
embedding_df = pd.DataFrame(embedding_loaded)

X = pd.concat([X_simple.reset_index(drop=True), embedding_df.reset_index(drop=True)], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_features)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data, categorical_feature=categorical_features)

params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'learning_rate': 0.03,
    'num_leaves': 64,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 5,
    'verbosity': -1,
    'device': 'gpu',
    'max_bin': 255
}

print("Training LightGBM...")
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],
    num_boost_round=500,
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(period=10)
    ]
)

y_pred = model.predict(X_test)
y_pred_labels = np.argmax(y_pred, axis=1)
acc = accuracy_score(y_test, y_pred_labels)
f1 = f1_score(y_test, y_pred_labels, average='weighted')
print(f"\nTest Accuracy: {acc:.4f}")
print(f"Test F1 Score: {f1:.4f}")

embed_lookup_df = pd.concat([df[['gmap_id']].reset_index(drop=True), embedding_df], axis=1)
gmap_id_to_embedding = embed_lookup_df.groupby('gmap_id').first().to_dict(orient='index')


Test Accuracy: 0.6183

Test F1 Score: 0.5009

In [ ]:
def evaluate_recommender_metrics(model, df, gmap_id_to_embedding,
                                 user_col='user_id', item_col='gmap_id',
                                 K_list=[5, 10, 20], num_negatives=99):
    print("Evaluating Recommender Metrics")

    user_items = defaultdict(set)
    for uid, gid in zip(df[user_col], df[item_col]):
        user_items[uid].add(gid)

    all_items = df[item_col].unique().tolist()
    test_users = df[user_col].unique()
    embedding_dim = len(next(iter(gmap_id_to_embedding.values())))

    results_by_k = {k: {'recall': [], 'precision': [], 'hit': [], 'ndcg': []} for k in K_list}

    for user in tqdm(test_users, desc="Users"):
        user_df = df[df[user_col] == user]
        if user_df.empty:
            continue
        pos_item = user_df.sample(1, random_state=42)[item_col].values[0]

        negatives = random.sample(
            [item for item in all_items if item not in user_items[user]],
            min(num_negatives, len(all_items) - len(user_items[user]) - 1)
        )
        candidates = [pos_item] + negatives
        candidate_labels = [1] + [0] * len(negatives)

        rows = []
        for item in candidates:
            row = user_df.iloc[0].copy()
            row[item_col] = item
            rows.append(row)

        test_batch = pd.DataFrame(rows)
        test_batch['time_of_day'] = test_batch['time_of_day'].astype('category')

        features = [
            'hour', 'weekday', 'month', 'is_weekend',
            'time_of_day', 'days_since_review',
            'text_length', 'num_exclamations'
        ]
        X_struct = test_batch[features].reset_index(drop=True)

        X_embed = pd.DataFrame([
            gmap_id_to_embedding.get(item, {i: 0 for i in range(embedding_dim)})
            for item in candidates
        ])
        X_batch = pd.concat([X_struct, X_embed], axis=1)

        scores = model.predict(X_batch)
        scores = scores[:, -1] if scores.ndim == 2 else scores

        ranked_indices = np.argsort(-scores)
        ranked_labels = np.array(candidate_labels)[ranked_indices]

        for k in K_list:
            top_k = ranked_labels[:k]
            hit = int(1 in top_k)
            rank = np.where(ranked_labels == 1)[0][0] + 1
            ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0
            precision = top_k.sum() / k
            recall = 1.0 if 1 in top_k else 0

            results_by_k[k]['recall'].append(recall)
            results_by_k[k]['precision'].append(precision)
            results_by_k[k]['hit'].append(hit)
            results_by_k[k]['ndcg'].append(ndcg)

    print("\n Ranking Metrics:")
    for k in K_list:
        recall = np.mean(results_by_k[k]['recall'])
        precision = np.mean(results_by_k[k]['precision'])
        hit = np.mean(results_by_k[k]['hit'])
        ndcg = np.mean(results_by_k[k]['ndcg'])
        print(f"@{k}: Recall={recall:.4f}, Precision={precision:.4f}, HitRate={hit:.4f}, NDCG={ndcg:.4f}")

evaluate_recommender_metrics(
    model=model,
    df=df,
    gmap_id_to_embedding=gmap_id_to_embedding,
    user_col='user_id',
    item_col='gmap_id',
    K_list=[5, 10, 20],
    num_negatives=99
)

Evaluating Recommender Metrics
Users: 100%|██████████| 221056/221056 [2:53:12<00:00, 21.27it/s]

Ranking Metrics:

@5: Recall=0.0410, Precision=0.0082, HitRate=0.0410, NDCG=0.0284

@10: Recall=0.0845, Precision=0.0085, HitRate=0.0845, NDCG=0.0423

@20: Recall=0.1712, Precision=0.0086, HitRate=0.1712, NDCG=0.0639

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


!pip install sentence-transformers tqdm

import torch
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# Load Dataset
file_path = '/content/drive/MyDrive/goodreads_interactions_comics_graphic.json'

with open(file_path, 'r') as f:
    lines = f.readlines()

data = [json.loads(line) for line in tqdm(lines, desc="Loading JSONL")]
df = pd.DataFrame(data)
df = df[df['review_text_incomplete'].str.strip() != ""]

# Aggregate Reviews per Book
book_reviews = df.groupby("book_id")["review_text_incomplete"].apply(lambda x: " ".join(x)).reset_index()

# Generate BERT Embeddings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentenceTransformer('all-mpnet-base-v2', device=device)

print("Generating book embeddings...")
book_embeddings = model.encode(
    book_reviews['review_text_incomplete'].tolist(),
    batch_size=64,
    convert_to_tensor=True,
    show_progress_bar=True,
    device=device
)

book_embeddings = book_embeddings.cpu()
book_reviews['embedding'] = [emb for emb in book_embeddings]
book_index = {book_id: idx for idx, book_id in enumerate(book_reviews['book_id'])}

# Cosine Similarity Recommendation
def recommend_books_from_user_profile(user_profile, top_k=5, exclude_ids=None):
    exclude_ids = set(exclude_ids or [])
    sim_scores = torch.nn.functional.cosine_similarity(user_profile, book_embeddings)


    for idx, book_id in enumerate(book_reviews['book_id']):
        if book_id in exclude_ids:
            sim_scores[idx] = -1

    topk = torch.topk(sim_scores, top_k)
    indices = topk.indices.cpu().numpy()
    return [book_reviews.iloc[i]['book_id'] for i in indices]

# Evaluation Metrics
def precision_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & set(relevant)) / len(relevant) if relevant else 0

def ndcg_at_k(recommended, relevant, k):
    dcg = sum([1 / np.log2(i + 2) if item in relevant else 0 for i, item in enumerate(recommended[:k])])
    idcg = sum([1 / np.log2(i + 2) for i in range(min(len(relevant), k))])
    return dcg / idcg if idcg > 0 else 0

def map_at_k(recommended, relevant, k):
    hits = 0
    sum_precisions = 0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            sum_precisions += hits / (i + 1)
    return sum_precisions / min(len(relevant), k) if relevant else 0

# Evaluation Loop
k = 10
precision_list, recall_list, ndcg_list, map_list = [], [], [], []

print("Evaluating recommender...")
for user_id, group in tqdm(df.groupby("user_id"), desc="Users"):
    if len(group) < 2:
        continue

    books = group['book_id'].tolist()
    test_book = books[-1]
    train_books = [b for b in books[:-1] if b in book_index]

    if not train_books or test_book not in book_index:
        continue

    train_embs = torch.stack([book_reviews.iloc[book_index[b]]['embedding'] for b in train_books])
    user_profile = train_embs.median(dim=0).values.unsqueeze(0).cpu()

    recommended = recommend_books_from_user_profile(user_profile, top_k=k, exclude_ids=train_books)
    relevant = [test_book]

    precision_list.append(precision_at_k(recommended, relevant, k))
    recall_list.append(recall_at_k(recommended, relevant, k))
    ndcg_list.append(ndcg_at_k(recommended, relevant, k))
    map_list.append(map_at_k(recommended, relevant, k))

# Final Metrics Report
print(f"\nEvaluation Results (k={k}):")
print(f"Precision@{k}: {np.mean(precision_list):.4f}")
print(f"Recall@{k}:    {np.mean(recall_list):.4f}")
print(f"NDCG@{k}:      {np.mean(ndcg_list):.4f}")
print(f"MAP@{k}:       {np.mean(map_list):.4f}")


Evaluation Results (k=10):

Precision@10: 0.0050

Recall@10:    0.0499

NDCG@10:      0.0299

MAP@10:       0.0238


#LightGBM + Vader GoogleMaps


In [ ]:

!pip install -q lightgbm tqdm
import nltk
nltk.download('vader_lexicon')
from google.colab import drive
drive.mount('/content/drive')
import json

import pandas as pd
import numpy as np
from tqdm import tqdm
import lightgbm as lgb
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import torch

# Check GPU
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

file_path = '/content/drive/MyDrive/review-South_Carolina_10.json'


with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]

df = pd.DataFrame(data)

# Check and handle missing columns
expected_cols = {'user_id', 'gmap_id', 'rating', 'time', 'text'}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in dataset: {missing}")

df['text'] = df['text'].fillna('')

# VADER Sentiment Analysis
sia = SentimentIntensityAnalyzer()

def vader_sentiment_batch(texts, batch_size=10000):
    sentiments = []
    for i in tqdm(range(0, len(texts), batch_size), desc="VADER Sentiment", ncols=100):
        batch = texts[i:i+batch_size]
        scores = [sia.polarity_scores(text)['compound'] for text in batch]
        sentiments.extend(scores)
    return sentiments

df['vader_sentiment'] = vader_sentiment_batch(df['text'].tolist())

# Time features
df['time'] = pd.to_datetime(df['time'], unit='ms', errors='coerce')
df['hour'] = df['time'].dt.hour
df['weekday'] = df['time'].dt.weekday
df['month'] = df['time'].dt.month

def time_of_day(hour):
    if pd.isna(hour):
        return 'unknown'
    if 5 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 17:
        return 'afternoon'
    elif 17 <= hour < 21:
        return 'evening'
    else:
        return 'night'

df['time_of_day'] = df['hour'].apply(time_of_day).astype('category')
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0)
latest_time = df['time'].max()
df['days_since_review'] = (latest_time - df['time']).dt.days

# Encode IDs
df['user_id'] = df['user_id'].astype('category')
df['gmap_id'] = df['gmap_id'].astype('category')

# Define features for LightGBM
features = [
    'user_id', 'gmap_id', 'hour', 'weekday', 'month',
    'is_weekend', 'time_of_day', 'days_since_review', 'vader_sentiment'
]

X = df[features]
y = df['rating']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train LightGBM
train_data = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=['user_id', 'gmap_id', 'time_of_day']
)
test_data = lgb.Dataset(
    X_test, label=y_test,
    reference=train_data,
    categorical_feature=['user_id', 'gmap_id', 'time_of_day']
)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'verbosity': -1
}

print("Training LightGBM...")
evals_result = {}
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],
    num_boost_round=100,
    callbacks=[
        lgb.early_stopping(stopping_rounds=10),
        lgb.log_evaluation(period=10)
    ]
)



# Evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(np.mean((y_test - y_pred)**2))
print(f"\nTest RMSE: {rmse:.4f}")

#  Feature importance
lgb.plot_importance(model, max_num_features=15, importance_type='gain')
plt.show()

from collections import defaultdict
from tqdm import tqdm

# Build interaction map from training set
train_user_items = df.loc[X_train.index].groupby('user_id')['gmap_id'].apply(set).to_dict()
all_items = df['gmap_id'].unique().tolist()
test_users = df.loc[X_test.index]['user_id'].unique()

K_list = [5, 10, 20]
results_by_k = {k: {'recall': [], 'precision': [], 'hit': [], 'ndcg': []} for k in K_list}

# Prepare categorical encodings if needed
X_test_full = X.copy()
X_test_full['rating'] = y

from collections import defaultdict
from tqdm import tqdm
import random

# Setup
K_list = [5, 10, 20]
results_by_k = {k: {'recall': [], 'precision': [], 'hit': [], 'ndcg': []} for k in K_list}
all_items = df['gmap_id'].unique().tolist()
train_user_items = df.loc[X_train.index].groupby('user_id')['gmap_id'].apply(set).to_dict()
test_users = df.loc[X_test.index]['user_id'].unique()

# Precompute features
user_feats = df[['user_id', 'hour', 'weekday', 'month', 'is_weekend', 'time_of_day', 'days_since_review']].drop_duplicates('user_id').set_index('user_id')
item_feats = df[['gmap_id', 'vader_sentiment']].drop_duplicates('gmap_id').set_index('gmap_id')

# Prepare candidate rows
candidate_rows = []
labels = []

print("\n Preparing candidate batch...")

for user_id in tqdm(test_users, desc="Users"):
    if user_id not in train_user_items:
        continue
    user_seen = train_user_items[user_id]
    user_df = X_test[X_test['user_id'] == user_id]
    if user_df.empty:
        continue

    pos_item = user_df.sample(1, random_state=42)['gmap_id'].values[0]
    negatives = [item for item in all_items if item not in user_seen and item != pos_item]
    if len(negatives) < 99:
        continue
    sampled_negs = random.sample(negatives, 99)

    candidate_items = [pos_item] + sampled_negs
    user_data = user_feats.loc[user_id]

    for item_id in candidate_items:
        row = {
            'user_id': user_id,
            'gmap_id': item_id,
            'hour': user_data['hour'],
            'weekday': user_data['weekday'],
            'month': user_data['month'],
            'is_weekend': user_data['is_weekend'],
            'time_of_day': user_data['time_of_day'],
            'days_since_review': user_data['days_since_review'],
            'vader_sentiment': item_feats.loc[item_id]['vader_sentiment'],
        }
        candidate_rows.append(row)
    labels.extend([1] + [0] * 99)

# Build batch dataframe
candidate_df = pd.DataFrame(candidate_rows)

# Match categorical types
for cat in ['user_id', 'gmap_id', 'time_of_day']:
    candidate_df[cat] = candidate_df[cat].astype(pd.api.types.CategoricalDtype(categories=df[cat].cat.categories))

print("\n Predicting on batch...")
scores = model.predict(candidate_df[features])
candidate_df['score'] = scores
candidate_df['label'] = labels

print("\n Evaluating...")
grouped = candidate_df.groupby('user_id')

for user_id, group in tqdm(grouped, desc="Scoring users"):
    group = group.sort_values('score', ascending=False)
    ranked_labels = group['label'].values

    for k in K_list:
        top_k = ranked_labels[:k]
        hit = int(1 in top_k)

        pos_indices = np.where(ranked_labels == 1)[0]
        if len(pos_indices) == 0:
            continue  # skip users with no positive item

        rank = pos_indices[0] + 1
        ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0
        precision = top_k.sum() / k
        recall = 1.0 if 1 in top_k else 0

        results_by_k[k]['recall'].append(recall)
        results_by_k[k]['precision'].append(precision)
        results_by_k[k]['hit'].append(hit)
        results_by_k[k]['ndcg'].append(ndcg)


print("\n Final Ranking Metrics:")
for k in K_list:
    print(f"@{k}: Recall={np.mean(results_by_k[k]['recall']):.4f}, "
          f"Precision={np.mean(results_by_k[k]['precision']):.4f}, "
          f"HitRate={np.mean(results_by_k[k]['hit']):.4f}, "
          f"NDCG={np.mean(results_by_k[k]['ndcg']):.4f}")




Final Ranking Metrics:

@5: Recall=0.2365, Precision=0.0473, HitRate=0.2365, NDCG=0.1416

@10: Recall=0.2880, Precision=0.0288, HitRate=0.2880, NDCG=0.1586

@20: Recall=0.3132, Precision=0.0157, HitRate=0.3132, NDCG=0.1648

#SASRec and Bert Hybrid Recommender Goodreads

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


!pip install sentence-transformers tqdm

import torch
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

file_path = '/content/drive/MyDrive/goodreads_interactions_comics_graphic.json'

with open(file_path, 'r') as f:
    lines = f.readlines()

data = [json.loads(line) for line in tqdm(lines, desc="Loading JSONL")]
df = pd.DataFrame(data)
df = df[df['review_text_incomplete'].str.strip() != ""]

# Aggregate Reviews per Book
book_reviews = df.groupby("book_id")["review_text_incomplete"].apply(lambda x: " ".join(x)).reset_index()

# Generate BERT Embeddings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentenceTransformer('all-mpnet-base-v2', device=device)

print("Generating book embeddings...")
book_embeddings = model.encode(
    book_reviews['review_text_incomplete'].tolist(),
    batch_size=64,
    convert_to_tensor=True,
    show_progress_bar=True,
    device=device
)

book_embeddings = book_embeddings.cpu()
book_reviews['embedding'] = [emb for emb in book_embeddings]
book_index = {book_id: idx for idx, book_id in enumerate(book_reviews['book_id'])}

# Cosine Similarity Recommendation
def recommend_books_from_user_profile(user_profile, top_k=5, exclude_ids=None):
    exclude_ids = set(exclude_ids or [])
    sim_scores = torch.nn.functional.cosine_similarity(user_profile, book_embeddings)


    for idx, book_id in enumerate(book_reviews['book_id']):
        if book_id in exclude_ids:
            sim_scores[idx] = -1

    topk = torch.topk(sim_scores, top_k)
    indices = topk.indices.cpu().numpy()
    return [book_reviews.iloc[i]['book_id'] for i in indices]

# Evaluation Metrics
def precision_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & set(relevant)) / len(relevant) if relevant else 0

def ndcg_at_k(recommended, relevant, k):
    dcg = sum([1 / np.log2(i + 2) if item in relevant else 0 for i, item in enumerate(recommended[:k])])
    idcg = sum([1 / np.log2(i + 2) for i in range(min(len(relevant), k))])
    return dcg / idcg if idcg > 0 else 0

def map_at_k(recommended, relevant, k):
    hits = 0
    sum_precisions = 0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            sum_precisions += hits / (i + 1)
    return sum_precisions / min(len(relevant), k) if relevant else 0

# Evaluation Loop
k = 10
precision_list, recall_list, ndcg_list, map_list = [], [], [], []

print("Evaluating recommender...")
for user_id, group in tqdm(df.groupby("user_id"), desc="Users"):
    if len(group) < 2:
        continue

    books = group['book_id'].tolist()
    test_book = books[-1]
    train_books = [b for b in books[:-1] if b in book_index]

    if not train_books or test_book not in book_index:
        continue

    train_embs = torch.stack([book_reviews.iloc[book_index[b]]['embedding'] for b in train_books])
    user_profile = train_embs.median(dim=0).values.unsqueeze(0).cpu()

    recommended = recommend_books_from_user_profile(user_profile, top_k=k, exclude_ids=train_books)
    relevant = [test_book]

    precision_list.append(precision_at_k(recommended, relevant, k))
    recall_list.append(recall_at_k(recommended, relevant, k))
    ndcg_list.append(ndcg_at_k(recommended, relevant, k))
    map_list.append(map_at_k(recommended, relevant, k))

# Final Metrics Report
print(f"\nEvaluation Results (k={k}):")
print(f"Precision@{k}: {np.mean(precision_list):.4f}")
print(f"Recall@{k}:    {np.mean(recall_list):.4f}")
print(f"NDCG@{k}:      {np.mean(ndcg_list):.4f}")
print(f"MAP@{k}:       {np.mean(map_list):.4f}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successful

Loading JSONL: 100%|██████████| 7347630/7347630 [00:32<00:00, 226954.82it/s]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

...

Generating book embeddings...


Batches:   0%|          | 0/1398 [00:00<?, ?it/s]

Evaluating recommender...


Users: 100%|██████████| 59513/59513 [1:36:24<00:00, 10.29it/s]



Evaluation Results (k=10):
Precision@10: 0.0050
Recall@10:    0.0499
NDCG@10:      0.0299
MAP@10:       0.0238


#Catboost Google Maps

In [ ]:
!pip install catboost scikit-learn pandas tqdm optuna

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, json, numpy as np, optuna, random
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm
from collections import defaultdict

file_path = '/content/drive/MyDrive/review-South_Carolina_10.json'
data = [json.loads(line) for line in open(file_path)]
df = pd.DataFrame(data)[['user_id', 'gmap_id', 'rating', 'time']].dropna()
df['timestamp'] = pd.to_datetime(df['time'], unit='ms')
df['time_norm'] = (df['time'] - df['time'].min()) / (df['time'].max() - df['time'].min())

# Feature engineering
user_stats = df.groupby('user_id').agg(
    user_num_reviews=('rating', 'count'),
    user_avg_rating=('rating', 'mean'),
    user_first_review=('timestamp', 'min')
).reset_index()
user_stats['user_days_since_first'] = (df['timestamp'].max() - user_stats['user_first_review']).dt.days

item_stats = df.groupby('gmap_id').agg(
    item_num_reviews=('rating', 'count'),
    item_avg_rating=('rating', 'mean'),
    item_first_review=('timestamp', 'min')
).reset_index()
item_stats['item_days_since_first'] = (df['timestamp'].max() - item_stats['item_first_review']).dt.days

df = df.merge(user_stats.drop(columns='user_first_review'), on='user_id', how='left')
df = df.merge(item_stats.drop(columns='item_first_review'), on='gmap_id', how='left')
df.drop(columns=['timestamp'], inplace=True)
df = df[df['user_num_reviews'] >= 3]
df = df[df['item_num_reviews'] >= 3]

# === 3. Split Dataset ===
train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

cat_features = ['user_id', 'gmap_id']
num_features = [
    'time_norm', 'user_num_reviews', 'user_avg_rating', 'user_days_since_first',
    'item_num_reviews', 'item_avg_rating', 'item_days_since_first'
]
all_features = cat_features + num_features

train_pool = Pool(train_df[all_features], label=train_df['rating'], cat_features=cat_features)
test_pool = Pool(test_df[all_features], cat_features=cat_features)

final_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.02,
    loss_function='RMSE',
    cat_features=cat_features,
    task_type="GPU",
    devices='0',
    verbose=0
)
final_model.fit(train_pool)
test_df['pred_rating'] = final_model.predict(test_df[all_features])
mse = mean_squared_error(test_df['rating'], test_df['pred_rating'])
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_df['rating'], test_df['pred_rating'])

print("\n=== Test Set Rating Metrics ===")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
K = 10
N_NEG = 99
all_items = set(df['gmap_id'].unique())
seen_items = train_val_df.groupby('user_id')['gmap_id'].apply(set).to_dict()
test_positives = test_df[test_df['rating'] >= 4].groupby('user_id').first().reset_index()

user_feats = df[['user_id', 'user_num_reviews', 'user_avg_rating', 'user_days_since_first']].drop_duplicates('user_id').set_index('user_id').to_dict('index')
item_feats = df[['gmap_id', 'item_num_reviews', 'item_avg_rating', 'item_days_since_first']].drop_duplicates('gmap_id').set_index('gmap_id').to_dict('index')

precision_list, recall_list, ndcg_list, hit_list = [], [], [], []

for _, row in tqdm(test_positives.iterrows(), total=len(test_positives)):
    user_id = row['user_id']
    pos_item = row['gmap_id']

    if user_id not in seen_items:
        continue

    user_seen = seen_items[user_id]
    negatives = list(all_items - user_seen - {pos_item})
    if len(negatives) < N_NEG:
        continue

    sampled_negs = random.sample(negatives, N_NEG)
    candidate_items = [pos_item] + sampled_negs

    records = []
    for item_id in candidate_items:
        user_f = user_feats.get(user_id)
        item_f = item_feats.get(item_id)
        if user_f and item_f:
            record = {
                'user_id': user_id,
                'gmap_id': item_id,
                'time_norm': 1.0,
                **user_f,
                **item_f
            }
            records.append(record)

    if len(records) < K:
        continue

    candidate_df = pd.DataFrame(records)
    scores = final_model.predict(candidate_df[all_features])
    candidate_df['score'] = scores

    top_k = candidate_df.sort_values('score', ascending=False).head(K)
    pred_items = top_k['gmap_id'].tolist()

    hit = int(pos_item in pred_items)
    precision = hit / K
    recall = 1.0 if hit else 0.0
    rank = pred_items.index(pos_item) + 1 if pos_item in pred_items else 0
    ndcg = 1 / np.log2(rank + 1) if rank > 0 else 0.0

    hit_list.append(hit)
    precision_list.append(precision)
    recall_list.append(recall)
    ndcg_list.append(ndcg)

print("\n=== Sampled Top-K Ranking Metrics ===")
print(f"Precision@{K}: {np.mean(precision_list):.4f}")
print(f"Recall@{K}:    {np.mean(recall_list):.4f}")
print(f"NDCG@{K}:      {np.mean(ndcg_list):.4f}")
print(f"Hit@{K}:       {np.mean(hit_list):.4f}")




Precision@10: 0.0034

Recall@10:    0.0344

NDCG@10:      0.0166

Hit@10:       0.0344